# 02 Conversational Ai Assistant

#### Settings

In [6]:
from litellm import completion
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

import gradio as gr

messages = [
    {"role":"user", "content":"Hello!"}
]

response = completion(
    model="ollama/llama3.1:8b",
    messages=messages,
    api_base="http://localhost:11434"
)

print(response.choices[0].message.content)


Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [7]:
import os
import ollama
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

import gradio as gr

In [8]:
load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")
ollama_api_key = None

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/"

In [9]:
messages = [
    {"role":"user", "content":"Hello!"}
]

In [10]:
client_gemini = OpenAI(base_url=gemini_url, api_key=google_api_key)

response_gemini = client_gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=messages)

In [11]:
display(Markdown(response_gemini.choices[0].message.content))

Hello there! How can I help you today?

#### Gradio Interface

In [12]:
def cheers(text):
    # print(f"Cheeeersss!!! {text} !!!")
    return text.upper() + "~~~~!!!"

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be cheered", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=cheers, 
    title="CHEERS!!!",
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["Hello", "wuhoo"],
    flagging_mode="never"
    )

# view.launch(auth=("user", "pwd"), auth_message="Please Insert Your Username and Password.")
view.launch()

In [62]:
import requests
from litellm import completion
import gradio as gr

def get_ollama_models():
    try:
        response = requests.get("http://localhost:11434/api/tags")
        if response.status_code == 200:
            models = [f"ollama/{m['name']}" for m in response.json().get("models", [])]
            return models if models else ["ollama/gemma3:270m"]
    except Exception:
        return ["ollama/gemma3:270m"]

messages = [
    {"role":"system", 
     "content": "You are a fun chat assistant! You will not only converse with the user, but also bring surprises in expressions and actions with emoji."}
]


def message_local_llama(user_prompt, history, model_name):
    global messages

    messages.append({"role":"user", "content": user_prompt})

    response = completion(
        model=model_name,
        messages=messages,
        base_url="http://localhost:11434",
        temperature=1,
        stream=True
    )

    # print(response)
    partial_messages = ""
    for chunk in response:
        content = chunk.choices[0].delta.content
        if content:
            partial_messages += content
            yield partial_messages


available_models = get_ollama_models()

chat_room = gr.ChatInterface(
    fn=message_local_llama,
    additional_inputs=[
        gr.Dropdown(choices=available_models, value=available_models[0], label="Choose a model")
    ],
    additional_inputs_accordion=gr.Accordion(label="Model Settings", open=True),
    type="messages"
)

chat_room.launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.
